# AWS RL Env — GRPO Training (multi-turn + parallel envs)

This notebook trains a Qwen2.5-Coder-3B policy on the AWS RL environment using **GRPO** with:

- **Multi-turn rollouts** — each task runs up to `MAX_TURNS` steps; each step is one `aws ...` command, the command's output is fed back into the next turn.
- **Parallel environments** — `NUM_GENERATIONS` MiniStack-backed env sessions run concurrently, all rolling out the *same* curriculum-picked task.
- **Curriculum** — `Curriculum.next_task()` picks one task per GRPO step; group-level reward feeds back via `Curriculum.record_result(...)` driving promotion + spaced repetition.
- **Optuna** — TPE search over learning rate, KL coefficient, num_generations, temperature, top-p, LoRA rank, and max_turns. Frozen held-out validation tasks evaluate each trial.

The heavy lifting lives in [`train_grpo.py`](./train_grpo.py); this notebook is a thin driver that mirrors `kube-sre-gym/kube_sre_gym_colab.ipynb`.


## 1 - Install dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q \
    "trl>=0.21" \
    "transformers>=4.45" \
    "peft>=0.13" \
    "datasets>=2.20" \
    "huggingface_hub>=0.24" \
    "websockets>=13" \
    "openenv-core[core]>=0.2.2" \
    "pyyaml>=6.0" \
    "matplotlib" \
    "optuna>=3.6" \
    "plotly" \
    "kaleido" \
    "httpx"
%pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"


## 2 - Configuration

Everything you'll typically tune lives in this cell.

In [ ]:
import os
from pathlib import Path
from datetime import datetime

# --- Environment server ---
ENV_URL = os.environ.get("AWS_RL_ENV_URL", "http://localhost:8000")

# --- Model & adapter ---
MODEL_ID    = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"
SFT_ADAPTER = "Sizzing/aws-rl-sft-qwen25coder3b-adapter"  # set to None to skip SFT init
HUB_REPO    = None  # e.g. "your-org/aws-rl-grpo-qwen25coder3b"

# --- Training defaults (Optuna may override) ---
NUM_GENERATIONS  = 8        # parallel envs == GRPO group size
MAX_TURNS        = 6        # multi-turn cap per episode
MAX_STEPS        = 200      # GRPO optimizer steps
MAX_TOTAL_TOKENS = 4096     # token budget per episode (anti-OOM)
MAX_PROMPT_LEN   = 2048
MAX_COMPL_LEN    = 256

# --- Optuna ---
RUN_OPTUNA          = True
N_TRIALS            = 6
TRIAL_MAX_STEPS     = 30
VAL_TASKS_PER_TIER  = 2

# --- Output ---
TIMESTAMP   = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
OUTPUT_DIR  = Path(f"outputs/aws-rl-grpo-{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Env URL    : {ENV_URL}")
print(f"Model      : {MODEL_ID}")
print(f"SFT adapter: {SFT_ADAPTER}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"Optuna     : {'on' if RUN_OPTUNA else 'off'} ({N_TRIALS} trials, {TRIAL_MAX_STEPS} steps each)")


## 3 - Authenticate to HF Hub (and optionally W&B)

In [ ]:
import os

# HF Hub
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except (ImportError, KeyError, ModuleNotFoundError):
    pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF Hub: logged in")
else:
    print("HF Hub: HF_TOKEN not set (push_to_hub will be disabled)")

# Optional: W&B
try:
    from google.colab import userdata
    os.environ.setdefault("WANDB_API_KEY", userdata.get("WANDB_API_KEY"))
except (ImportError, KeyError, ModuleNotFoundError):
    pass


## 4 - Smoke-test the env URL

In [ ]:
import httpx

resp = httpx.get(f"{ENV_URL}/health", timeout=10.0)
print(f"GET {ENV_URL}/health -> {resp.status_code}")
print(resp.text[:500])
assert resp.status_code == 200, "env server is not responding — start it before training"


## 5 - Imports from `train_grpo`

All heavy logic (rollout, env pool, reward funcs, Optuna search, training loop) lives in `train_grpo.py` at the repo root.

In [ ]:
import json
import logging
from pathlib import Path

from train_grpo import (
    SYSTEM_PROMPT,
    DEFAULT_CFG,
    SamplingCfg,
    load_policy,
    MultiTurnEnvPool,
    plot_rewards,
    pick_validation_task_ids,
    evaluate_on_validation,
    optuna_search,
    run_training,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")
print("System prompt (first 200 chars):")
print(SYSTEM_PROMPT[:200])


## 6 - Pick fixed validation task ids

A frozen list of tasks (k per tier) used as the held-out set across **all** Optuna trials and post-training comparisons. Stored to disk for reproducibility.

In [ ]:
val_task_ids = pick_validation_task_ids(k_per_tier=VAL_TASKS_PER_TIER, seed=42)
val_path = OUTPUT_DIR / "val_task_ids.json"
val_path.write_text(json.dumps(val_task_ids))
print(f"Validation task ids ({len(val_task_ids)}): {val_task_ids}")
print(f"Saved to {val_path}")


## 7 - Optuna hyperparameter search

Set `RUN_OPTUNA=False` in the config cell to skip and use `DEFAULT_CFG`.

In [ ]:
best_cfg = None

if RUN_OPTUNA:
    study = optuna_search(
        n_trials=N_TRIALS,
        trial_max_steps=TRIAL_MAX_STEPS,
        val_task_ids=val_task_ids,
        base_model=MODEL_ID,
        sft_adapter=SFT_ADAPTER,
        env_url=ENV_URL,
        output_dir=OUTPUT_DIR,
        max_total_tokens=MAX_TOTAL_TOKENS,
        max_completion_length=MAX_COMPL_LEN,
        max_prompt_length=MAX_PROMPT_LEN,
    )
    best_cfg = {**DEFAULT_CFG, **dict(study.best_params)}
    print(f"\nBest objective : {study.best_value:.4f}")
    print(f"Best params    : {dict(study.best_params)}")
else:
    print("Skipping Optuna; using DEFAULT_CFG.")
    best_cfg = dict(DEFAULT_CFG)

with open(OUTPUT_DIR / "best_cfg.json", "w") as f:
    json.dump(best_cfg, f, indent=2)
print(f"Saved best_cfg -> {OUTPUT_DIR / 'best_cfg.json'}")


In [ ]:
# Optional Optuna visualisations (skip silently if Optuna wasn't run)
if RUN_OPTUNA:
    try:
        import optuna.visualization as vis
        import plotly.io as pio
        pio.renderers.default = "notebook"
        vis.plot_optimization_history(study).show()
        vis.plot_param_importances(study).show()
    except Exception as e:
        print(f"(visualisation skipped: {e})")


## 8 - Final GRPO training pass with the best config

In [ ]:
print(f"Final config: {best_cfg}")

# Override via best_cfg, falling back to top-of-notebook defaults
NUM_GENERATIONS = int(best_cfg["num_generations"])
MAX_TURNS       = int(best_cfg["max_turns"])

run_training(
    cfg=best_cfg,
    base_model=MODEL_ID,
    sft_adapter=SFT_ADAPTER,
    env_url=ENV_URL,
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    max_total_tokens=MAX_TOTAL_TOKENS,
    max_completion_length=MAX_COMPL_LEN,
    max_prompt_length=MAX_PROMPT_LEN,
    push_to_hub=False,
    hub_repo=HUB_REPO,
)


## 9 - Reward curves

`plot_rewards` reads `reward_log.csv` (written incrementally by `EpisodeLogger`), so the chart is meaningful even if training was interrupted.

In [ ]:
from IPython.display import Image, display

reward_csv = OUTPUT_DIR / "reward_log.csv"
plot_path = OUTPUT_DIR / "reward_plot.png"
plot_rewards(reward_csv, plot_path)
if plot_path.exists():
    display(Image(filename=str(plot_path)))
else:
    print("No plot generated (no rows in reward_log.csv).")


## 10 - Quick post-training validation re-run (optional)

Run the same held-out tasks again on the freshly trained adapter and compare to whatever each Optuna trial achieved on the same set.

In [ ]:
# Re-load policy in inference mode
model, tokenizer = load_policy(MODEL_ID, SFT_ADAPTER, trainable=False)
pool = MultiTurnEnvPool(ENV_URL, size=1)
pool.start()

sampling = SamplingCfg(
    temperature=float(best_cfg["temperature"]),
    top_p=float(best_cfg["top_p"]),
    max_new_tokens=MAX_COMPL_LEN,
    max_prompt_length=MAX_PROMPT_LEN,
)

try:
    metrics = evaluate_on_validation(
        model=model,
        tokenizer=tokenizer,
        pool=pool,
        val_task_ids=val_task_ids,
        system_prompt=SYSTEM_PROMPT,
        max_turns=int(best_cfg["max_turns"]),
        max_total_tokens=MAX_TOTAL_TOKENS,
        sampling=sampling,
    )
    print(f"Post-training validation metrics: {metrics}")
    with open(OUTPUT_DIR / "post_train_val.json", "w") as f:
        json.dump(metrics, f, indent=2)
finally:
    pool.close()


## 11 - Push to Hugging Face Hub (optional)

In [ ]:
# Uncomment to push the trained adapter:
#
# from huggingface_hub import create_repo, upload_folder
# create_repo(HUB_REPO, exist_ok=True, private=False)
# upload_folder(folder_path=str(OUTPUT_DIR), repo_id=HUB_REPO, repo_type="model")
# print(f"Pushed: https://huggingface.co/{HUB_REPO}")
